# Cleaning files

### 1. Imports & Paths


In [1]:
import json
from pathlib import Path

import pandas as pd


In [2]:
from pathlib import Path

# RAW files (original JSON/TXT) live next to the notebook
RAW_DIR = Path(".")

# CLEANED files go into /data
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

RAW_DIR, DATA_DIR


(WindowsPath('.'), WindowsPath('data'))

### 2. Clean TaladRod data

#### Load the raw JSON

In [3]:
# Path to your raw data file
DATA_DIR = Path(".")  / "data"
TALADROD_PATH = RAW_DIR / "taladrodData.json"    
TALADROD_PATH



WindowsPath('taladrodData.json')

In [5]:
# Load JSON file into Python list
with TALADROD_PATH.open("r", encoding="utf-8") as f:
    taladrod_raw = json.load(f)

# Convert to a DataFrame
df_raw = pd.DataFrame(taladrod_raw)





#### Inspect

In [6]:
df_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 948 entries, 0 to 947
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   year      883 non-null    object
 1   brand     883 non-null    object
 2   serie     877 non-null    object
 3   color     883 non-null    object
 4   gear      883 non-null    object
 5   mileage   878 non-null    object
 6   price     935 non-null    object
 7   location  818 non-null    object
dtypes: object(8)
memory usage: 59.4+ KB


In [7]:
df_raw.columns.tolist()

['year', 'brand', 'serie', 'color', 'gear', 'mileage', 'price', 'location']

In [8]:
df_raw.head(3)

,year,brand,serie,color,gear,mileage,price,location
0,2023,HONDA,CRV,ขาว,เกียร์ออโต้,20000,1259000,กรุงเทพฯ
1,2023,TOYOTA,YARIS ATIV,เทา,เกียร์ออโต้,51000,469000,กรุงเทพฯ
2,2016,MINI,COUPE,แดง,เกียร์ออโต้,106000,680000,กรุงเทพฯ


In [9]:
df_raw.iloc[0].to_dict()


{'year': '2023',
 'brand': 'HONDA',
 'serie': 'CRV',
 'color': 'ขาว',
 'gear': 'เกียร์ออโต้',
 'mileage': '20000',
 'price': '1259000',
 'location': 'กรุงเทพฯ'}

#### Cell checking 

In [10]:
# How many rows?
len(df_raw)

948

In [11]:
# Count of non-null values per column
df_raw.count()


year        883
brand       883
serie       877
color       883
gear        883
mileage     878
price       935
location    818
dtype: int64

In [12]:
# Unique brands / sample
df_raw["brand"].value_counts().head(20)


brand
TOYOTA        233
HONDA         131
BENZ          111
BMW            84
MAZDA          44
FORD           42
ISUZU          40
NISSAN         40
MITSUBISHI     33
HYUNDAI        16
PORSCHE        12
VOLVO          11
SUZUKI         11
MG             11
AUDI           10
MINI           10
VOLKSWAGEN      9
CHEVROLET       7
SUBARU          6
KIA             4
Name: count, dtype: int64

In [13]:
# Sample of 'serie', 'year', 'mileage', 'price'
df_raw[["brand", "serie", "year", "mileage", "price"]].head(10)

,brand,serie,year,mileage,price
0,HONDA,CRV,2023,20000,1259000
1,TOYOTA,YARIS ATIV,2023,51000,469000
2,MINI,COUPE,2016,106000,680000
3,BMW,X3,2022,64000,1999000
4,FORD,EVEREST,2023,35000,1299000
5,CHEVROLET,CAPTIVA,2013,180000,158000
6,MAZDA,CX-30,2020,100000,589000
7,TOYOTA,CAMRY,2016,182000,488000
8,MITSUBISHI,TRITON 18-23,2020,59990,355000
9,ISUZU,MU-X,2019,129661,839000


#### Clean function

In [14]:
def clean_taladrod_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Take raw TaladRod DataFrame and return a cleaned, standardized version.
    """

    # 1. Rename / standardize columns
    df = df.rename(
        columns={
            "serie": "model",           # treat 'serie' as model/series
            "mileage": "mileage_km",
            "price": "price_thb",
        }
    )

    # Keep only the columns we care about
    keep_cols = [
        "brand",
        "model",
        "year",
        "price_thb",
        "mileage_km",
        "color",
        "gear",
        "location",
    ]
    # Only keep columns that actually exist
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    # 2. Strip whitespace from string columns
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()

    # 3. Normalize brand + model text
    if "brand" in df.columns:
        df["brand"] = df["brand"].str.upper()

    if "model" in df.columns:
        df["model"] = df["model"].str.strip()

    # 4. Clean year
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        # keep only reasonable car years
        df = df[df["year"].between(1980, 2025, inclusive="both")]

    # 5. Clean price_thb
    if "price_thb" in df.columns:
        df["price_thb"] = (
            df["price_thb"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("฿", "", regex=False)
            .str.strip()
        )
        df["price_thb"] = pd.to_numeric(df["price_thb"], errors="coerce")
        # keep prices in a plausible THB range (tweak later if needed)
        df = df[df["price_thb"].between(10_000, 20_000_000)]

    # 6. Clean mileage_km
    if "mileage_km" in df.columns:
        df["mileage_km"] = (
            df["mileage_km"]
            .astype(str)
            .str.lower()
            .str.replace(",", "", regex=False)
            .str.replace("km", "", regex=False)
            .str.strip()
        )
        df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")
        # keep non-negative mileages; allow NaN for unknown
        df = df[df["mileage_km"].isna() | (df["mileage_km"] >= 0)]

    # 7. Add source column
    df["source"] = "taladrod"

    # 8. Reset index
    df = df.reset_index(drop=True)

    return df


In [81]:
df_clean["year"] = df_clean["year"].astype("Int64")
#cast year to int

#### Apply cleaning function to file

In [15]:
df_clean = clean_taladrod_df(df_raw)
df_clean.head()


,brand,model,year,price_thb,mileage_km,color,gear,location,source
0,HONDA,CRV,2023.0,1259000,20000.0,ขาว,เกียร์ออโต้,กรุงเทพฯ,taladrod
1,TOYOTA,YARIS ATIV,2023.0,469000,51000.0,เทา,เกียร์ออโต้,กรุงเทพฯ,taladrod
2,MINI,COUPE,2016.0,680000,106000.0,แดง,เกียร์ออโต้,กรุงเทพฯ,taladrod
3,BMW,X3,2022.0,1999000,64000.0,ขาว,เกียร์ออโต้,กรุงเทพฯ,taladrod
4,FORD,EVEREST,2023.0,1299000,35000.0,น้ำตาล,เกียร์ออโต้,None,taladrod


In [16]:
#Check how many rows survived:
len(df_raw), len(df_clean)


(948, 883)

In [17]:
#Check distributions
df_clean[["year", "price_thb", "mileage_km"]].describe()


,year,price_thb,mileage_km
count,883.000000,8.830000e+02,878.000000
mean,2017.103058,8.447792e+05,111063.678815
std,4.834872,1.062403e+06,68349.554683
min,1991.000000,6.800000e+04,1500.000000
25%,2014.000000,3.450000e+05,60000.000000
50%,2018.000000,5.650000e+05,100000.000000
75%,2021.000000,9.570000e+05,150000.000000
max,2025.000000,1.370000e+07,430000.000000


In [18]:
#brand breakdown 
df_clean["brand"].value_counts().head(20)


brand
TOYOTA        233
HONDA         131
BENZ          111
BMW            84
MAZDA          44
FORD           42
ISUZU          40
NISSAN         40
MITSUBISHI     33
HYUNDAI        16
PORSCHE        12
VOLVO          11
SUZUKI         11
MG             11
AUDI           10
MINI           10
VOLKSWAGEN      9
CHEVROLET       7
SUBARU          6
KIA             4
Name: count, dtype: int64

In [19]:
df_clean.sample(10, random_state=1)
#looking at a random sample

,brand,model,year,price_thb,mileage_km,color,gear,location,source
320,BMW,SERIES 3,2021.0,1350000,89000.0,ดำ,เกียร์ออโต้,กรุงเทพฯ,taladrod
523,MAZDA,MAZDA3,2022.0,615000,59363.0,เทา,เกียร์ออโต้,กรุงเทพฯ,taladrod
519,PORSCHE,CAYMAN,2013.0,2899000,79800.0,เหลือง,เกียร์ออโต้,กรุงเทพฯ,taladrod
3,BMW,X3,2022.0,1999000,64000.0,ขาว,เกียร์ออโต้,กรุงเทพฯ,taladrod
613,HONDA,CITY,2020.0,429000,56000.0,ขาว,เกียร์ออโต้,กรุงเทพฯ,taladrod
593,TOYOTA,CAMRY,2019.0,699000,122672.0,ดำ,เกียร์ออโต้,กรุงเทพฯ,taladrod
711,TOYOTA,CAMRY,2020.0,920000,31789.0,แดง,เกียร์ออโต้,กรุงเทพฯ,taladrod
2,MINI,COUPE,2016.0,680000,106000.0,แดง,เกียร์ออโต้,กรุงเทพฯ,taladrod
189,MAZDA,CX-5,2018.0,428000,130529.0,ดำ,เกียร์ออโต้,กรุงเทพฯ,taladrod
34,PEUGEOT,3008,2020.0,719000,88000.0,ขาว,เกียร์ออโต้,กรุงเทพฯ,taladrod


#### Save cleaned output

In [20]:
#save cleaned output
OUTPUT_PATH = DATA_DIR / "taladrod_clean.csv"
df_clean.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH


WindowsPath('data/taladrod_clean.csv')

## 3. Clean One2Car data

#### Load raw

In [21]:
#Clean one2car file 
ONE2CAR_PATH = Path("./one2carData.json")  

with ONE2CAR_PATH.open("r", encoding="utf-8") as f:
    one2car_raw = json.load(f)

type(one2car_raw), len(one2car_raw)


(list, 338)

In [22]:
pd.DataFrame(one2car_raw).head()
#See what im working with


,rawData,brand,serie,section,version,year,mileage,price,location,link,origin,date
0,2012 Toyota INNOVA 2.0 (ปี 04-11) G Wagon,Toyota,INNOVA,2.0 G Wagon,04-11,2012,107000,236000,กรุงเทพมหานคร,www.one2car.com/for-sale/toyota-innova-g-%E0%B...,one2car,2025-09-09T09:20:55.551Z
1,2018 Mazda 3 2.0 (ปี 14-18) S Sports Hatchback,Mazda,3,2.0 S Sports Hatchback,14-18,2018,175000,365000,ประจวบคีรีขันธ์,www.one2car.com/for-sale/mazda-3-s-sports-%E0%...,one2car,2025-09-09T09:20:55.552Z
2,2014 Mitsubishi TRITON 2.5 DOUBLE CAB (ปี 05-1...,Mitsubishi,TRITON,2.5 DOUBLE CABGLS Plus Pickup - Double Cab AT,05-15,2014,190000,220000,อุดรธานี,www.one2car.com/for-sale/mitsubishi-triton-gls...,one2car,2025-09-09T09:20:55.552Z
3,2023 Honda HR-V 1.5 (ปี 22-28) e:HEV RS SUV - ...,Honda,HR-V,1.5 e:HEV RS SUV - e HEV RS,22-28,2023,80000,829000,กรุงเทพมหานคร,www.one2car.com/for-sale/honda-hr-v-e-hev-rs-%...,one2car,2025-09-09T09:20:55.553Z
4,2013 Ford RANGER 2.2 DOUBLE CAB (ปี 12-15) XLT...,Ford,RANGER,2.2 DOUBLE CABXLT Hi-Rider Pickup - XLT Double...,12-15,2013,285000,259000,กรุงเทพมหานคร,www.one2car.com/for-sale/ford-ranger-xlt-doubl...,one2car,2025-09-09T09:20:55.553Z


In [23]:
ONE2CAR_PATH = RAW_DIR / "one2carData.json"    
ONE2CAR_PATH, ONE2CAR_PATH.exists()


(WindowsPath('one2carData.json'), True)

In [24]:
with ONE2CAR_PATH.open("r", encoding="utf-8") as f:
    one2car_raw = json.load(f)

df_one2_raw = pd.DataFrame(one2car_raw)
df_one2_raw.head()


,rawData,brand,serie,section,version,year,mileage,price,location,link,origin,date
0,2012 Toyota INNOVA 2.0 (ปี 04-11) G Wagon,Toyota,INNOVA,2.0 G Wagon,04-11,2012,107000,236000,กรุงเทพมหานคร,www.one2car.com/for-sale/toyota-innova-g-%E0%B...,one2car,2025-09-09T09:20:55.551Z
1,2018 Mazda 3 2.0 (ปี 14-18) S Sports Hatchback,Mazda,3,2.0 S Sports Hatchback,14-18,2018,175000,365000,ประจวบคีรีขันธ์,www.one2car.com/for-sale/mazda-3-s-sports-%E0%...,one2car,2025-09-09T09:20:55.552Z
2,2014 Mitsubishi TRITON 2.5 DOUBLE CAB (ปี 05-1...,Mitsubishi,TRITON,2.5 DOUBLE CABGLS Plus Pickup - Double Cab AT,05-15,2014,190000,220000,อุดรธานี,www.one2car.com/for-sale/mitsubishi-triton-gls...,one2car,2025-09-09T09:20:55.552Z
3,2023 Honda HR-V 1.5 (ปี 22-28) e:HEV RS SUV - ...,Honda,HR-V,1.5 e:HEV RS SUV - e HEV RS,22-28,2023,80000,829000,กรุงเทพมหานคร,www.one2car.com/for-sale/honda-hr-v-e-hev-rs-%...,one2car,2025-09-09T09:20:55.553Z
4,2013 Ford RANGER 2.2 DOUBLE CAB (ปี 12-15) XLT...,Ford,RANGER,2.2 DOUBLE CABXLT Hi-Rider Pickup - XLT Double...,12-15,2013,285000,259000,กรุงเทพมหานคร,www.one2car.com/for-sale/ford-ranger-xlt-doubl...,one2car,2025-09-09T09:20:55.553Z


#### Inspect

In [25]:
#inspect
df_one2_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 338 entries, 0 to 337
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   rawData   338 non-null    object
 1   brand     338 non-null    object
 2   serie     338 non-null    object
 3   section   336 non-null    object
 4   version   315 non-null    object
 5   year      338 non-null    object
 6   mileage   338 non-null    object
 7   price     338 non-null    object
 8   location  338 non-null    object
 9   link      338 non-null    object
 10  origin    338 non-null    object
 11  date      338 non-null    object
dtypes: object(12)
memory usage: 31.8+ KB


In [26]:
df_one2_raw[["brand", "serie", "section", "version", "year", "mileage", "price", "location"]].head(10)


,brand,serie,section,version,year,mileage,price,location
0,Toyota,INNOVA,2.0 G Wagon,04-11,2012,107000,236000,กรุงเทพมหานคร
1,Mazda,3,2.0 S Sports Hatchback,14-18,2018,175000,365000,ประจวบคีรีขันธ์
2,Mitsubishi,TRITON,2.5 DOUBLE CABGLS Plus Pickup - Double Cab AT,05-15,2014,190000,220000,อุดรธานี
3,Honda,HR-V,1.5 e:HEV RS SUV - e HEV RS,22-28,2023,80000,829000,กรุงเทพมหานคร
4,Ford,RANGER,2.2 DOUBLE CABXLT Hi-Rider Pickup - XLT Double...,12-15,2013,285000,259000,กรุงเทพมหานคร
5,Toyota,Corolla Cross,1.8 Hybrid Premium Safety GR Sport SUV,20-26,2022,95000,879000,กรุงเทพมหานคร
6,Honda,HR-V,1.5 e:HEV EL SUV - e HEV EL,22-28,2022,76817,769000,กรุงเทพมหานคร
7,BMW,730Li,3.0 E 66 Sedan,01-08,2003,110000,249000,กรุงเทพมหานคร
8,Toyota,Corolla Altis,1.6 E Sedan,08-13,2012,5000,129000,กรุงเทพมหานคร
9,Honda,JAZZ,1.5 S Hatchback,14-22,2018,160000,359000,กรุงเทพมหานคร


#### Clean function

In [27]:
def clean_one2car_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Take raw One2Car DataFrame and return a cleaned, standardized version
    compatible with the TaladRod schema.
    """

    # 1. Rename columns to match our standard names
    df = df.rename(
        columns={
            "serie": "model",
            "mileage": "mileage_km",
            "price": "price_thb",
            "origin": "source_raw",  # keep original origin separately if we want
        }
    )

    # 2. Keep only the core columns we need right now
    keep_cols = [
        "brand",
        "model",
        "year",
        "price_thb",
        "mileage_km",
        "location",
        "rawData",
        "section",
        "version",
        "link",
        "date",
        "source_raw",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]
    df = df[keep_cols].copy()

    # 3. Strip whitespace from strings
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()

    # 4. Normalize brand + model
    if "brand" in df.columns:
        df["brand"] = df["brand"].str.upper()

    if "model" in df.columns:
        df["model"] = df["model"].str.strip()

    # 5. Clean year
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        df = df[df["year"].between(1980, 2025, inclusive="both")]

    # 6. Clean price_thb
    if "price_thb" in df.columns:
        df["price_thb"] = (
            df["price_thb"]
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("฿", "", regex=False)
            .str.strip()
        )
        df["price_thb"] = pd.to_numeric(df["price_thb"], errors="coerce")
        df = df[df["price_thb"].between(10_000, 20_000_000)]

    # 7. Clean mileage_km
    if "mileage_km" in df.columns:
        df["mileage_km"] = (
            df["mileage_km"]
            .astype(str)
            .str.lower()
            .str.replace(",", "", regex=False)
            .str.replace("km", "", regex=False)
            .str.strip()
        )
        df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")
        df = df[df["mileage_km"].isna() | (df["mileage_km"] >= 0)]

    # 8. Add missing columns for compatibility with TaladRod
    # (color and gear are unknown here)
    if "color" not in df.columns:
        df["color"] = pd.NA
    if "gear" not in df.columns:
        df["gear"] = pd.NA

    # 9. Set a clean 'source' column
    df["source"] = "one2car"

    # 10. Reset index
    df = df.reset_index(drop=True)

    return df


In [28]:
df_one2_clean = clean_one2car_df(df_one2_raw)
df_one2_clean.head()


,brand,model,year,price_thb,mileage_km,location,rawData,section,version,link,date,source_raw,color,gear,source
0,TOYOTA,INNOVA,2012,236000.0,107000,กรุงเทพมหานคร,2012 Toyota INNOVA 2.0 (ปี 04-11) G Wagon,2.0 G Wagon,04-11,www.one2car.com/for-sale/toyota-innova-g-%E0%B...,2025-09-09T09:20:55.551Z,one2car,<NA>,<NA>,one2car
1,MAZDA,3,2018,365000.0,175000,ประจวบคีรีขันธ์,2018 Mazda 3 2.0 (ปี 14-18) S Sports Hatchback,2.0 S Sports Hatchback,14-18,www.one2car.com/for-sale/mazda-3-s-sports-%E0%...,2025-09-09T09:20:55.552Z,one2car,<NA>,<NA>,one2car
2,MITSUBISHI,TRITON,2014,220000.0,190000,อุดรธานี,2014 Mitsubishi TRITON 2.5 DOUBLE CAB (ปี 05-1...,2.5 DOUBLE CABGLS Plus Pickup - Double Cab AT,05-15,www.one2car.com/for-sale/mitsubishi-triton-gls...,2025-09-09T09:20:55.552Z,one2car,<NA>,<NA>,one2car
3,HONDA,HR-V,2023,829000.0,80000,กรุงเทพมหานคร,2023 Honda HR-V 1.5 (ปี 22-28) e:HEV RS SUV - ...,1.5 e:HEV RS SUV - e HEV RS,22-28,www.one2car.com/for-sale/honda-hr-v-e-hev-rs-%...,2025-09-09T09:20:55.553Z,one2car,<NA>,<NA>,one2car
4,FORD,RANGER,2013,259000.0,285000,กรุงเทพมหานคร,2013 Ford RANGER 2.2 DOUBLE CAB (ปี 12-15) XLT...,2.2 DOUBLE CABXLT Hi-Rider Pickup - XLT Double...,12-15,www.one2car.com/for-sale/ford-ranger-xlt-doubl...,2025-09-09T09:20:55.553Z,one2car,<NA>,<NA>,one2car


### check size n basic stats

In [29]:
len(df_one2_raw), len(df_one2_clean)


(338, 337)

In [30]:
df_one2_clean[["year", "price_thb", "mileage_km"]].describe()


,year,price_thb,mileage_km
count,337.000000,3.370000e+02,337.000000
mean,2017.795252,8.443970e+05,96118.988131
std,4.800444,7.680543e+05,69454.772999
min,1988.000000,3.950000e+04,1.000000
25%,2016.000000,3.690000e+05,40000.000000
50%,2019.000000,6.390000e+05,90000.000000
75%,2021.000000,9.900000e+05,140000.000000
max,2025.000000,4.990000e+06,318546.000000


In [31]:
##sample data
df_one2_clean.sample(10, random_state=42)


,brand,model,year,price_thb,mileage_km,location,rawData,section,version,link,date,source_raw,color,gear,source
46,HONDA,CR-V,2019,655000.0,5000,กรุงเทพมหานคร,2019 Honda CR-V 1.6 (ปี 17-21) DT E SUV,1.6 DT E SUV,17-21,www.one2car.com/for-sale/honda-cr-v-dt-e-%E0%B...,2025-09-09T09:20:57.604Z,one2car,<NA>,<NA>,one2car
163,MAZDA,3,2018,438000.0,94613,นนทบุรี,2018 Mazda 3 2.0 (ปี 14-18) S Sports Hatchback,2.0 S Sports Hatchback,14-18,www.one2car.com/for-sale/mazda-3-s-sports-%E0%...,2025-09-09T09:21:08.516Z,one2car,<NA>,<NA>,one2car
57,TOYOTA,HILUX REVO,2023,759000.0,30000,กรุงเทพมหานคร,2023 Toyota HILUX REVO 2.4 DOUBLE CAB Prerunne...,2.4 DOUBLE CAB Prerunner Mid Pickup,None,www.one2car.com/for-sale/toyota-hilux-revo-pre...,2025-09-09T09:20:58.694Z,one2car,<NA>,<NA>,one2car
25,BMW,218i,2018,459000.0,115000,กรุงเทพมหานคร,2018 BMW 218i 1.5 F45 (ปี 14-21) Active Tourer...,1.5 F 45 Active Tourer Hatchback,14-21,www.one2car.com/for-sale/bmw-218i-active-toure...,2025-09-09T09:20:55.555Z,one2car,<NA>,<NA>,one2car
113,MAZDA,2,2021,369000.0,5000,อุดรธานี,2021 Mazda 2 1.3 (ปี 15-25) S Leather Sedan,1.3 S Leather Sedan,15-25,www.one2car.com/for-sale/mazda-2-s-leather-%E0...,2025-09-09T09:21:04.067Z,one2car,<NA>,<NA>,one2car
198,BMW,530e,2021,959000.0,120000,กรุงเทพมหานคร,2021 BMW 530e 2.0 G30 (ปี 17-22) Elite Sedan,2.0 G 30 Elite Sedan,17-22,www.one2car.com/for-sale/bmw-530e-elite-%E0%B8...,2025-09-09T09:21:10.392Z,one2car,<NA>,<NA>,one2car
108,MASERATI,GRANTURISMO,2009,3290000.0,40000,กรุงเทพมหานคร,2009 Maserati GRANTURISMO 4.2 (ปี 08-15) Coupe,4.2 Coupe,08-15,www.one2car.com/for-sale/maserati-granturismo-...,2025-09-09T09:21:04.067Z,one2car,<NA>,<NA>,one2car
63,ISUZU,D-MAX,2016,319000.0,129000,กรุงเทพมหานคร,2016 Isuzu D-MAX 1.9 SPACE CAB (ปี 11-17) Hi-L...,1.9 SPACE CABHi-Lander L Pickup - Space Cab,11-17,www.one2car.com/for-sale/isuzu-d-max-hi-lander...,2025-09-09T09:20:58.694Z,one2car,<NA>,<NA>,one2car
249,BMW,420i,2014,990000.0,100000,กรุงเทพมหานคร,2014 BMW 420i 2.0 F32 (ปี 13-17) M Sport Coupe...,2.0 F 32 M Sport Coupe - F 32,13-17,www.one2car.com/for-sale/bmw-420i-m-sport-%E0%...,2025-09-09T09:21:14.702Z,one2car,<NA>,<NA>,one2car
326,TOYOTA,Yaris Ativ,2024,414000.0,20000,กรุงเทพมหานคร,2024 Toyota Yaris Ativ 1.2 (ปี 22-28) Sport Sedan,1.2 Sport Sedan,22-28,www.one2car.com/for-sale/toyota-yaris-ativ-spo...,2025-09-09T09:21:21.406Z,one2car,<NA>,<NA>,one2car


#### Save cleaned output

In [32]:
ONE2CAR_OUTPUT_PATH = DATA_DIR / "one2car_clean.csv"
ONE2CAR_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_one2_clean.to_csv(ONE2CAR_OUTPUT_PATH, index=False)
ONE2CAR_OUTPUT_PATH


WindowsPath('data/one2car_clean.csv')

## 4. Clean Motto data

#### Load raw

In [33]:
MOTTO_PATH = RAW_DIR / "motto-data.json"

with MOTTO_PATH.open("r", encoding="utf-8") as f:
    motto_raw = json.load(f)

motto_raw


[{'lot': 'ลำดับที่ 1001',
  'data': '2020 โตโยต้า YARIS 1.2 MID',
  'grade': 'C',
  'engine': '1200cc',
  'gear': 'ออโตเมติก',
  'colour': 'Grey',
  'mileage': '91498',
  'location': 'กรุงเทพมหานคร',
  'type': 'G',
  'status': 'Sold',
  'price': '278000',
  'date': '2025-10-27',
  '_id': 'X1FBK257|ลำดับที่ 1001'},
 {'lot': 'ลำดับที่ 1002',
  'data': '2023 อีซูซุ MU-X 1.9 ULTIMATE',
  'grade': 'A',
  'engine': '1900cc',
  'gear': 'ออโตเมติก',
  'colour': 'Blue',
  'mileage': '33788',
  'location': 'สุพรรณบุรี',
  'type': 'G',
  'status': 'Sold',
  'price': '790000',
  'date': '2025-10-27',
  '_id': 'X1FBK257|ลำดับที่ 1002'},
 {'lot': 'ลำดับที่ 1003',
  'data': '2024 อีซูซุ D-MAX 1.9 SPACECAB L DA HI-LANDER',
  'grade': 'S',
  'engine': '1900cc',
  'gear': 'ออโตเมติก',
  'colour': 'Grey',
  'mileage': '8253',
  'location': 'กรุงเทพมหานคร',
  'type': 'G',
  'status': 'Sold',
  'price': '478000',
  'date': '2025-10-27',
  '_id': 'X1FBK257|ลำดับที่ 1003'},
 {'lot': 'ลำดับที่ 1004',
  'data'

#### inspect

In [35]:
import re

# Where the raw files live
RAW_DIR = Path(".")          # or change to your raw folder if needed

# Where cleaned CSVs will be saved
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
DATA_DIR

WindowsPath('data')

In [36]:
MOTTO_PATH = RAW_DIR / "motto-data.json"
MOTTO_PATH, MOTTO_PATH.exists()


(WindowsPath('motto-data.json'), True)

In [37]:
df_motto_raw = pd.DataFrame(motto_raw)
df_motto_raw.head()


,lot,data,grade,engine,gear,colour,mileage,location,type,status,price,date,_id
0,ลำดับที่ 1001,2020 โตโยต้า YARIS 1.2 MID,C,1200cc,ออโตเมติก,Grey,91498,กรุงเทพมหานคร,G,Sold,278000,2025-10-27,X1FBK257|ลำดับที่ 1001
1,ลำดับที่ 1002,2023 อีซูซุ MU-X 1.9 ULTIMATE,A,1900cc,ออโตเมติก,Blue,33788,สุพรรณบุรี,G,Sold,790000,2025-10-27,X1FBK257|ลำดับที่ 1002
2,ลำดับที่ 1003,2024 อีซูซุ D-MAX 1.9 SPACECAB L DA HI-LANDER,S,1900cc,ออโตเมติก,Grey,8253,กรุงเทพมหานคร,G,Sold,478000,2025-10-27,X1FBK257|ลำดับที่ 1003
3,ลำดับที่ 1004,2025 อีซูซุ D-MAX 1.9 X-SERIES SPEED SPACECAB,A,1900cc,ธรรมดา,Black,6536,กรุงเทพมหานคร,G,Sold,422000,2025-10-27,X1FBK257|ลำดับที่ 1004
4,ลำดับที่ 1005,2021 อีซูซุ D-MAX 1.9 S SPACECAB,C,1900cc,ธรรมดา,Grey,23004,กรุงเทพมหานคร,G,Sold,290000,2025-10-27,X1FBK257|ลำดับที่ 1005


In [38]:
df_motto_raw.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   lot       126 non-null    object
 1   data      126 non-null    object
 2   grade     126 non-null    object
 3   engine    126 non-null    object
 4   gear      126 non-null    object
 5   colour    126 non-null    object
 6   mileage   126 non-null    object
 7   location  126 non-null    object
 8   type      126 non-null    object
 9   status    126 non-null    object
 10  price     126 non-null    object
 11  date      126 non-null    object
 12  _id       126 non-null    object
dtypes: object(13)
memory usage: 12.9+ KB


In [106]:
df_motto_raw[["lot", "data", "grade", "engine", "gear", "colour", "mileage", "location", "type", "price", "date"]].head(10)


,lot,data,grade,engine,gear,colour,mileage,location,type,price,date
0,1004,2011 Isuzu D-Max Spark 2.5 EX,P,2500cc,Manual,Grey,181983,Krabi,Y,0,18/03/2025
1,1008,2018 Mini COOPER S COUNTRYMAN 2.0 AUTO,F,2000cc,Automatic,Sky-White,37279,Bangkok,Y,0,18/03/2025
2,1020,2012 Mercedes-Benz C 180 CGI,F,1800cc,Automatic,White-Black,108192,Songkhla,R,0,18/03/2025
3,1021,2008 Honda CIVIC 1.8 S I-VTEC,X,1800cc,Automatic,Grey,227905,Krabi,R,0,18/03/2025
4,1023,2020 Mini COOPER S COUNTRYMAN RHD 2.0 AUTO,A,2000cc,Automatic,Red-White,118733,Bangkok,Y,0,18/03/2025
5,1024,2008 Toyota Fortuner 2.7 V VVT-I,P,2700cc,Automatic,Black,479894,Songkhla,Y,0,18/03/2025
6,1025,2015 Isuzu D-Max Spark 2.5 S,P,2500cc,Manual,White,482027,Bangkok,R,0,18/03/2025
7,1032,2017 Mazda CX-5 SKYACTIV 2.2 XDL AUTO AWD,F,2200cc,Automatic,Black,121418,Krabi,R,0,18/03/2025
8,1036,2010 Isuzu D-MAX 2.5 CAB4 HI-LANDER,F,2500cc,Manual,Grey,245907,Satun,Y,0,18/03/2025
9,1043,2017 Isuzu D-MAX SPARK 1.9 S,F,1900cc,Manual,Grey-Black,154462,Bangkok,R,0,18/03/2025


#### clean

In [40]:
def clean_motto_df(df_raw: pd.DataFrame) -> pd.DataFrame:
    df = df_raw.copy()

    # --- Basic renames ---
    df = df.rename(columns={
        "data": "raw_title",
        "colour": "color",
    })

    # --- Lot number (numeric) ---
    df["lot_num"] = (
        df["lot"]
        .astype(str)
        .str.extract(r"(\d+)")
        .astype("Int64")  # pandas nullable int
    )

    # --- Price numeric ---
    df["price_thb"] = (
        df["price"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": None, "nan": None})
    )
    df["price_thb"] = pd.to_numeric(df["price_thb"], errors="coerce")

    # --- Mileage numeric (km) ---
    df["mileage_km"] = (
        df["mileage"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": None, "nan": None})
    )
    df["mileage_km"] = pd.to_numeric(df["mileage_km"], errors="coerce")

    # --- Year from the raw title (assumes first 4 digits are year) ---
    df["year"] = (
        df["raw_title"]
        .astype(str)
        .str.extract(r"(\d{4})")[0]
    )
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    # --- Very simple brand mapping from Thai text in raw_title ---
    # This is just a starting point – you can expand later
    brand_map = {
        "โตโยต้า": "TOYOTA",
        "โตโยต้า ": "TOYOTA",
        "TOYOTA": "TOYOTA",
        "ฮอนด้า": "HONDA",
        "HONDA": "HONDA",
        "มาสด้า": "MAZDA",
        "MAZDA": "MAZDA",
        "อีซูซุ": "ISUZU",
        "ISUZU": "ISUZU",
        "นิสสัน": "NISSAN",
        "NISSAN": "NISSAN",
        "มิตซูบิชิ": "MITSUBISHI",
        "ฟอร์ด": "FORD",
        "FORD": "FORD",
        "เชฟโรเลต": "CHEVROLET",
        "CHEVROLET": "CHEVROLET",
        "เบนซ์": "MERCEDES-BENZ",
        "บีเอ็ม": "BMW",
        "BMW": "BMW",
    }

    def extract_brand(title: str) -> str | None:
        if not isinstance(title, str):
            return None
        # remove year at the start
        t = re.sub(r"^\s*\d{4}\s*", "", title)
        for th, eng in brand_map.items():
            if th in t:
                return eng
        return None

    df["brand"] = df["raw_title"].apply(extract_brand)

    # --- Simple model text (everything after year and brand, very rough) ---
    def extract_model(title: str) -> str | None:
        if not isinstance(title, str):
            return None
        t = re.sub(r"^\s*\d{4}\s*", "", title)
        # remove known brand tokens
        for th in brand_map.keys():
            t = t.replace(th, "")
        return t.strip() or None

    df["model_raw"] = df["raw_title"].apply(extract_model)

    # --- Normalize gear text a bit (optional) ---
    # We'll keep original gear but also a simplified "gear_type"
    gear_text = df["gear"].astype(str)

    df["gear_type"] = None
    df.loc[gear_text.str.contains("ออโต", case=False), "gear_type"] = "A/T"
    df.loc[gear_text.str.contains("Auto", case=False), "gear_type"] = "A/T"
    df.loc[gear_text.str.contains("ธรรมดา", case=False), "gear_type"] = "M/T"
    df.loc[gear_text.str.contains("Man", case=False), "gear_type"] = "M/T"

    # --- Location clean ---
    df["location"] = df["location"].astype(str).str.strip()

    # --- Source tag ---
    df["source"] = "motto"

    # --- Final column order ---
    cols = [
        "brand",
        "model_raw",
        "year",
        "price_thb",
        "mileage_km",
        "color",
        "gear",
        "gear_type",
        "location",
        "grade",
        "engine",
        "type",
        "status",
        "date",
        "lot_num",
        "raw_title",
        "source",
    ]

    # Keep only columns that actually exist (in case some are missing)
    cols_existing = [c for c in cols if c in df.columns]
    df_clean = df[cols_existing].copy()

    return df_clean


In [41]:
df_motto_clean = clean_motto_df(df_motto_raw)

df_motto_clean.head()


,brand,model_raw,year,price_thb,mileage_km,color,gear,gear_type,location,grade,engine,type,status,date,lot_num,raw_title,source
0,TOYOTA,YARIS 1.2 MID,2020,278000.0,91498,Grey,ออโตเมติก,A/T,กรุงเทพมหานคร,C,1200cc,G,Sold,2025-10-27,1001,2020 โตโยต้า YARIS 1.2 MID,motto
1,ISUZU,MU-X 1.9 ULTIMATE,2023,790000.0,33788,Blue,ออโตเมติก,A/T,สุพรรณบุรี,A,1900cc,G,Sold,2025-10-27,1002,2023 อีซูซุ MU-X 1.9 ULTIMATE,motto
2,ISUZU,D-MAX 1.9 SPACECAB L DA HI-LANDER,2024,478000.0,8253,Grey,ออโตเมติก,A/T,กรุงเทพมหานคร,S,1900cc,G,Sold,2025-10-27,1003,2024 อีซูซุ D-MAX 1.9 SPACECAB L DA HI-LANDER,motto
3,ISUZU,D-MAX 1.9 X-SERIES SPEED SPACECAB,2025,422000.0,6536,Black,ธรรมดา,M/T,กรุงเทพมหานคร,A,1900cc,G,Sold,2025-10-27,1004,2025 อีซูซุ D-MAX 1.9 X-SERIES SPEED SPACECAB,motto
4,ISUZU,D-MAX 1.9 S SPACECAB,2021,290000.0,23004,Grey,ธรรมดา,M/T,กรุงเทพมหานคร,C,1900cc,G,Sold,2025-10-27,1005,2021 อีซูซุ D-MAX 1.9 S SPACECAB,motto


In [42]:
len(df_motto_raw), len(df_motto_clean)


(126, 126)

In [110]:
df_motto_clean[["year", "price_thb", "mileage_km"]].describe()


,year,price_thb,mileage_km
count,15.000000,0.0,15.000000
mean,2012.800000,NaN,200543.133333
std,3.913347,NaN,134661.552790
min,2008.000000,NaN,0.000000
25%,2009.500000,NaN,120075.500000
50%,2012.000000,NaN,181983.000000
75%,2016.000000,NaN,234439.500000
max,2020.000000,NaN,482027.000000


In [43]:
df_motto_clean.sample(10, random_state=1)


,brand,model_raw,year,price_thb,mileage_km,color,gear,gear_type,location,grade,engine,type,status,date,lot_num,raw_title,source
120,None,Subaru XV 2.0 I-P AWD,2021,NaN,0,Blue,ออโตเมติก,A/T,กรุงเทพมหานคร,N/A,2000cc,S,,2025-10-27,5043,2021 Subaru XV 2.0 I-P AWD,motto
46,TOYOTA,FORTUNER 2.4 V,2021,764000.0,73149,Black,ออโตเมติก,A/T,นครสวรรค์,B,2400cc,G,Sold,2025-10-27,1047,2021 โตโยต้า FORTUNER 2.4 V,motto
73,TOYOTA,HILUX REVO 2.4 E Z EDITION SMART CAB,2020,220000.0,237239,White,ธรรมดา,M/T,กรุงเทพมหานคร,E,2400cc,G,Not Sold,2025-10-27,1074,2020 โตโยต้า HILUX REVO 2.4 E Z EDITION SMART CAB,motto
90,None,NETA V -,2024,NaN,0,White,ออโตเมติก,A/T,ชัยภูมิ,N/A,95HP,S,,2025-10-27,5013,2024 NETA V -,motto
102,TOYOTA,VIOS 1.5 E VVT-I,2010,NaN,0,Grey,ออโตเมติก,A/T,กรุงเทพมหานคร,N/A,1500cc,S,,2025-10-27,5025,2010 โตโยต้า VIOS 1.5 E VVT-I,motto
53,TOYOTA,COMMUTER 2.5,2013,292000.0,454698,White,ธรรมดา,M/T,นครปฐม,E,2500cc,G,Sold,2025-10-27,1054,2013 โตโยต้า COMMUTER 2.5,motto
119,TOYOTA,วีออส 1.5 J VVT-I,2008,NaN,0,Black,ออโตเมติก,A/T,กรุงเทพมหานคร,N/A,1500cc,S,,2025-10-27,5042,2008 โตโยต้า วีออส 1.5 J VVT-I,motto
124,None,Mercedes-Benz E300 BLUETEC HYBRID 2.1,2013,NaN,0,White,ออโตเมติก,A/T,กรุงเทพมหานคร,N/A,2100cc,S,,2025-10-27,5047,2013 Mercedes-Benz E300 BLUETEC HYBRID 2.1,motto
116,ISUZU,ดีแม็กซ์ 1.9 DDI Z-PRESTIGE HI-LANDER CAB4 AT,2018,NaN,0,Black,ออโตเมติก,A/T,ขอนแก่น,N/A,1900cc,S,,2025-10-27,5039,2018 อีซูซุ ดีแม็กซ์ 1.9 DDI Z-PRESTIGE HI-LAN...,motto
42,TOYOTA,HILUX REVO 2.4 ENTRY Z EDITION SMART CAB,2023,310000.0,43581,Grey,ธรรมดา,M/T,อุทัยธานี,B,2400cc,G,Sold,2025-10-27,1043,2023 โตโยต้า HILUX REVO 2.4 ENTRY Z EDITION SM...,motto


#### save

In [44]:
MOTTO_OUT_PATH = DATA_DIR / "motto_clean.csv"
df_motto_clean.to_csv(MOTTO_OUT_PATH, index=False)
MOTTO_OUT_PATH



WindowsPath('data/motto_clean.csv')

## 5. Clean AppleAuction data

#### Load raw

In [45]:

APPLE_PATH = RAW_DIR / "Appleauction.txt"   # raw file in same folder as notebook
APPLE_PATH, APPLE_PATH.exists()


(WindowsPath('Appleauction.txt'), True)

#### inspect

In [46]:
with APPLE_PATH.open("r", encoding="utf-8") as f:
    apple_raw = f.readlines()

apple_raw[:10]   # just to peek


['TOYOTA,86 GT,86 GT 2.0 STD,2012,-,2560000,694000,747000\n',
 'TOYOTA,86 GT,86 GT 2.0 STD Man,2012,-,2490000,676000,729000\n',
 'TOYOTA,86 GT,86 GT 2.0 Top,2012,-,2740000,745500,800500\n',
 'TOYOTA,ALPHARD,ALPHARD 2.4 Hybrid,2013,-,3459000,604500,657500\n',
 'TOYOTA,ALPHARD,ALPHARD 2.4 V (MNC),2013,AT,3339000,578500,631500\n',
 'TOYOTA,ALPHARD,ALPHARD 3.5 V (MNC),2013,-,4099000,647000,700500\n',
 'TOYOTA,AVANZA,AVANZA 1.5 E (MY12) Man,2013,MT,569000,114500,127000\n',
 'TOYOTA,AVANZA,AVANZA 1.5 G (MY12),2013,AT,659000,131500,153000\n',
 'TOYOTA,AVANZA,AVANZA 1.5 S (MNC),2013,-,699000,143000,158500\n',
 'TOYOTA,AVANZA,AVANZA 1.5 S (MY12),2013,AT,699000,127000,164000\n']

In [47]:
# Parse AppleAuction lines into a raw DataFrame

import pandas as pd

rows = []

for line in apple_raw:
    # Remove surrounding quotes and newline, then strip spaces
    line = line.strip().strip("'").strip()
    if not line:
        continue

    parts = line.split(",")

    # Expect at least 8 parts: brand, model, subtype, year, gear, price1, price2, price3
    if len(parts) < 8:
        continue

    brand   = parts[0].strip().upper()
    model   = parts[1].strip()
    subtype = parts[2].strip()
    year    = parts[3].strip()
    gear    = parts[4].strip()
    price1  = parts[5].strip()
    price2  = parts[6].strip()
    price3  = parts[7].strip()

    rows.append(
        {
            "brand": brand,
            "model": model,
            "subtype": subtype,
            "year": year,
            "gear": gear,
            "price1": price1,
            "price2": price2,
            "price3": price3,
        }
    )

df_apple_raw = pd.DataFrame(rows)
df_apple_raw.head()


,brand,model,subtype,year,gear,price1,price2,price3
0,TOYOTA,86 GT,86 GT 2.0 STD,2012,-,2560000,694000,747000
1,TOYOTA,86 GT,86 GT 2.0 STD Man,2012,-,2490000,676000,729000
2,TOYOTA,86 GT,86 GT 2.0 Top,2012,-,2740000,745500,800500
3,TOYOTA,ALPHARD,ALPHARD 2.4 Hybrid,2013,-,3459000,604500,657500
4,TOYOTA,ALPHARD,ALPHARD 2.4 V (MNC),2013,AT,3339000,578500,631500


#### clean

In [48]:
# Clean AppleAuction data into standard schema

import numpy as np

def clean_appleauction_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Year
    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df[df["year"].between(1980, 2025, inclusive="both")]

    # Gear
    df["gear_clean"] = df["gear"].replace("-", pd.NA).str.upper()

    # Prices → numeric
    for col in ["price1", "price2", "price3"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Use average of available prices as price_thb
    df["price_thb"] = df[["price1", "price2", "price3"]].mean(axis=1, skipna=True)

    # AppleAuction has no mileage / color / location
    df["mileage_km"] = pd.NA
    df["color"] = pd.NA
    df["location"] = pd.NA

    # Final gear column
    df["gear"] = df["gear_clean"]

    # Source tag
    df["source"] = "appleauction"

    # Keep columns consistent with other datasets (plus extra Apple fields)
    keep_cols = [
        "brand",
        "model",
        "year",
        "price_thb",
        "mileage_km",
        "color",
        "gear",
        "location",
        "subtype",
        "price1",
        "price2",
        "price3",
        "source",
    ]
    df = df[keep_cols].reset_index(drop=True)

    return df


#### save

In [49]:
# 6.4 Apply cleaning and inspect

df_apple_clean = clean_appleauction_df(df_apple_raw)
df_apple_clean.head()


,brand,model,year,price_thb,mileage_km,color,gear,location,subtype,price1,price2,price3,source
0,TOYOTA,86 GT,2012.0,1.333667e+06,<NA>,<NA>,<NA>,<NA>,86 GT 2.0 STD,2560000,694000,747000,appleauction
1,TOYOTA,86 GT,2012.0,1.298333e+06,<NA>,<NA>,<NA>,<NA>,86 GT 2.0 STD Man,2490000,676000,729000,appleauction
2,TOYOTA,86 GT,2012.0,1.428667e+06,<NA>,<NA>,<NA>,<NA>,86 GT 2.0 Top,2740000,745500,800500,appleauction
3,TOYOTA,ALPHARD,2013.0,1.573667e+06,<NA>,<NA>,<NA>,<NA>,ALPHARD 2.4 Hybrid,3459000,604500,657500,appleauction
4,TOYOTA,ALPHARD,2013.0,1.516333e+06,<NA>,<NA>,AT,<NA>,ALPHARD 2.4 V (MNC),3339000,578500,631500,appleauction


In [50]:
APPLE_OUTPUT_PATH = DATA_DIR / "appleauction_clean.csv"
df_apple_clean.to_csv(APPLE_OUTPUT_PATH, index=False)
APPLE_OUTPUT_PATH


WindowsPath('data/appleauction_clean.csv')

## 8. Summary of outputs


In [58]:
# MERGE ALL DATASETS TOGETHER

import pandas as pd

# Load all cleaned files
df_tala   = pd.read_csv(DATA_DIR / "taladrod_clean.csv")
df_one2   = pd.read_csv(DATA_DIR / "one2car_clean.csv")
df_motto  = pd.read_csv(DATA_DIR / "motto_clean.csv")          # once fixed price
df_apple  = pd.read_csv(DATA_DIR / "appleauction_clean.csv")

# List them for merging
dfs = [df_tala, df_one2, df_motto, df_apple]

# Standardized base columns we expect
base_cols = [
    "brand",
    "model",
    "year",
    "price_thb",
    "mileage_km",
    "color",
    "gear",
    "location",
    "source"
]

# Ensure all datasets have all required columns
for i, df in enumerate(dfs):
    for col in base_cols:
        if col not in df.columns:
            df[col] = pd.NA
    dfs[i] = df[base_cols]   # reorder columns to match

# Merge everything
df_all = pd.concat(dfs, ignore_index=True, sort=False)

# Sanity check
print("Total rows:", len(df_all))
print(df_all['source'].value_counts())
df_all.head()


Total rows: 6805
source
appleauction    5459
taladrod         883
one2car          337
motto            126
Name: count, dtype: int64


,brand,model,year,price_thb,mileage_km,color,gear,location,source
0,HONDA,CRV,2023.0,1259000.0,20000.0,ขาว,เกียร์ออโต้,กรุงเทพฯ,taladrod
1,TOYOTA,YARIS ATIV,2023.0,469000.0,51000.0,เทา,เกียร์ออโต้,กรุงเทพฯ,taladrod
2,MINI,COUPE,2016.0,680000.0,106000.0,แดง,เกียร์ออโต้,กรุงเทพฯ,taladrod
3,BMW,X3,2022.0,1999000.0,64000.0,ขาว,เกียร์ออโต้,กรุงเทพฯ,taladrod
4,FORD,EVEREST,2023.0,1299000.0,35000.0,น้ำตาล,เกียร์ออโต้,NaN,taladrod


In [59]:
MASTER_OUTPUT_PATH = DATA_DIR / "all_cars_master.csv"
df_all.to_csv(MASTER_OUTPUT_PATH, index=False)
MASTER_OUTPUT_PATH


WindowsPath('data/all_cars_master.csv')

# Merge Motto + MottoSpan

In [61]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

df_motto = pd.read_csv(DATA_DIR / "motto_clean.csv")
df_span  = pd.read_csv(DATA_DIR / "motto_span_clean.csv")

# Make sure lot_num is same type on both sides
df_motto["lot_num"] = df_motto["lot_num"].astype("Int64")
df_span["lot_num"]  = df_span["lot_num"].astype("Int64")

df_motto_enriched = df_motto.merge(df_span, on="lot_num", how="left")

df_motto_enriched.head()


,brand,model_raw,year,price_thb,mileage_km,color,gear,gear_type,location,grade,...,started,reset,sold,winner_id,sold_amount,num_bids,num_hall_bids,num_internet_bids,min_bid,max_bid
0,TOYOTA,YARIS 1.2 MID,2020,278000.0,91498,Grey,ออโตเมติก,A/T,กรุงเทพมหานคร,C,...,True,True,True,838.0,278000.0,30.0,12.0,18.0,220000.0,278000.0
1,ISUZU,MU-X 1.9 ULTIMATE,2023,790000.0,33788,Blue,ออโตเมติก,A/T,สุพรรณบุรี,A,...,True,False,True,5608.0,790000.0,1.0,0.0,1.0,790000.0,790000.0
2,ISUZU,D-MAX 1.9 SPACECAB L DA HI-LANDER,2024,478000.0,8253,Grey,ออโตเมติก,A/T,กรุงเทพมหานคร,S,...,True,False,True,845.0,478000.0,45.0,24.0,21.0,390000.0,478000.0
3,ISUZU,D-MAX 1.9 X-SERIES SPEED SPACECAB,2025,422000.0,6536,Black,ธรรมดา,M/T,กรุงเทพมหานคร,A,...,True,False,True,845.0,422000.0,0.0,0.0,0.0,NaN,NaN
4,ISUZU,D-MAX 1.9 S SPACECAB,2021,290000.0,23004,Grey,ธรรมดา,M/T,กรุงเทพมหานคร,C,...,True,False,True,836.0,290000.0,36.0,5.0,31.0,180000.0,290000.0


In [62]:
MOTTO_ENRICHED_PATH = DATA_DIR / "motto_enriched.csv"
df_motto_enriched.to_csv(MOTTO_ENRICHED_PATH, index=False)
MOTTO_ENRICHED_PATH


WindowsPath('data/motto_enriched.csv')

# MASTER MERGE

In [4]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

df_taladrod = pd.read_csv(DATA_DIR / "taladrod_clean.csv")
df_one2car  = pd.read_csv(DATA_DIR / "one2car_clean.csv")
df_apple    = pd.read_csv(DATA_DIR / "appleauction_clean.csv")
df_motto    = pd.read_csv(DATA_DIR / "motto_enriched.csv")  # <-- use enriched here

# Assign channels
df_taladrod["channel"] = "marketplace"
df_one2car["channel"]  = "marketplace"
df_apple["channel"]    = "auction"
df_motto["channel"]    = "auction"

# Make sure 'source' exists
if "source" not in df_taladrod.columns:
    df_taladrod["source"] = "taladrod"
if "source" not in df_one2car.columns:
    df_one2car["source"] = "one2car"
if "source" not in df_apple.columns:
    df_apple["source"] = "appleauction"
if "source" not in df_motto.columns:
    df_motto["source"] = "motto"

COMMON_COLS = [
    "brand",
    "model",
    "model_raw",
    "year",
    "price_thb",
    "mileage_km",
    "color",
    "gear_type",
    "location",
    "lot_num",
    "grade",
    "engine",
    "type",
    "date",
    "sold_amount",        # from span
    "num_bids",           # from span
    "num_hall_bids",      # from span
    "num_internet_bids",  # from span
    "min_bid",
    "max_bid",
    "source",
    "channel",
]

def normalize(df):
    df = df.copy()
    # Ensure columns exist
    for col in COMMON_COLS:
        if col not in df.columns:
            df[col] = pd.NA

    # If model is missing but model_raw exists, use that
    if "model" in df.columns and "model_raw" in df.columns:
        df["model"] = df["model"].fillna(df["model_raw"])

    return df[COMMON_COLS]

df_taladrod_norm = normalize(df_taladrod)
df_one2car_norm  = normalize(df_one2car)
df_apple_norm    = normalize(df_apple)
df_motto_norm    = normalize(df_motto)

df_master = pd.concat(
    [df_taladrod_norm, df_one2car_norm, df_apple_norm, df_motto_norm],
    ignore_index=True,
    sort=False
)

MASTER_OUTPUT_PATH = DATA_DIR / "master_cars_all_sources.csv"
df_master.to_csv(MASTER_OUTPUT_PATH, index=False)
MASTER_OUTPUT_PATH


C:\Users\Zenbook\AppData\Local\Temp\ipykernel_19420\2897839575.py:70: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_master = pd.concat(


WindowsPath('data/master_cars_all_sources.csv')

In [5]:
import re
import pandas as pd

# quick peek that df_master is there
df_master.head()


,brand,model,model_raw,year,price_thb,mileage_km,color,gear_type,location,lot_num,...,type,date,sold_amount,num_bids,num_hall_bids,num_internet_bids,min_bid,max_bid,source,channel
0,HONDA,CRV,NaN,2023.0,1259000.0,20000.0,ขาว,NaN,กรุงเทพฯ,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,taladrod,marketplace
1,TOYOTA,YARIS ATIV,NaN,2023.0,469000.0,51000.0,เทา,NaN,กรุงเทพฯ,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,taladrod,marketplace
2,MINI,COUPE,NaN,2016.0,680000.0,106000.0,แดง,NaN,กรุงเทพฯ,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,taladrod,marketplace
3,BMW,X3,NaN,2022.0,1999000.0,64000.0,ขาว,NaN,กรุงเทพฯ,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,taladrod,marketplace
4,FORD,EVEREST,NaN,2023.0,1299000.0,35000.0,น้ำตาล,NaN,NaN,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,taladrod,marketplace


## parser for model_clean, series, trim

In [6]:
THAI_BRAND_TOKENS = {
    "TOYOTA":   ["โตโยต้า", "โตโยตา"],
    "HONDA":    ["ฮอนด้า", "ฮอนดา"],
    "MAZDA":    ["มาสด้า", "มาสดา"],
    "NISSAN":   ["นิสสัน"],
    "ISUZU":    ["อีซูซุ"],
    "FORD":     ["ฟอร์ด"],
    "MITSUBISHI": ["มิตซูบิชิ", "มิตซู"],
    "SUZUKI":   ["ซูซูกิ"],
    "MG":       ["เอ็มจี"],
    "HYUNDAI":  ["ฮุนได"],
    "VOLVO":    ["วอลโว่", "โวลโว่"],
    "KIA":      ["เกีย"],
    "MERCEDES": ["เบนซ์", "เมอร์เซเดส"],
    "BMW":      ["บีเอ็ม", "บีเอ็มดับเบิลยู"],
    "CHEVROLET": ["เชฟโรเลต", "เชฟ"],
}

def _safe_str(val) -> str:
    """Convert value to string; treat NA/None as empty string."""
    if pd.isna(val):
        return ""
    return str(val)

def parse_model_fields(row):
    """
    Use model_raw (and brand) to guess:
      - model_clean
      - series
      - trim

    Does NOT touch df_master['model'] directly: just returns fields.
    """
    raw            = _safe_str(row.get("model_raw")).strip()
    existing_model = _safe_str(row.get("model")).strip()
    brand          = _safe_str(row.get("brand")).strip().upper()

    # base result: use None instead of pd.NA (avoids dtype issues in .apply)
    result = {
        "model_clean": existing_model or None,
        "series":      None,
        "trim":        None,
    }

    if not raw:
        return pd.Series(result)

    # ---- normalise text ----
    text = raw.upper()

    # remove years like 2018, 2020, etc.
    text = re.sub(r"\b(19|20)\d{2}\b", " ", text)
    # remove things like "2.0", "1.8"
    text = re.sub(r"\b\d+\.\d+\b", " ", text)
    # remove engine cc / hp patterns like "2500CC", "204HP"
    text = re.sub(r"\b\d{2,4}(CC|HP)\b", " ", text)

    # strip English brand if present
    if brand:
        text = text.replace(brand, " ")

    # strip Thai brand variants
    for _, toks in THAI_BRAND_TOKENS.items():
        for t in toks:
            if not t:
                continue
            text = text.replace(t.upper(), " ")

    # split into tokens
    tokens = [t for t in re.split(r"[ ,/()\-]+", text) if t]
    if not tokens:
        return pd.Series(result)

    # ---------- model_clean ----------
    model_token = tokens[0]
    model_clean = existing_model if existing_model else model_token.title()
    result["model_clean"] = model_clean

    # ---------- series (brand-specific heuristics) ----------
    series = None
    upper_tokens = tokens

    if brand == "BMW":
        # e.g. "320 D 3 SERIES" -> "3 Series"
        for i in range(len(upper_tokens) - 1):
            if upper_tokens[i + 1] == "SERIES" and upper_tokens[i].isdigit():
                series = f"{upper_tokens[i]} Series".title()
                break
        if series is None and re.match(r"^[A-Z]\d$", upper_tokens[0]):
            series = upper_tokens[0].title()

    elif brand == "TOYOTA":
        joined = " ".join(upper_tokens)
        if "HILUX" in upper_tokens:
            if "REVO" in upper_tokens:
                series = "Hilux Revo"
            elif "VIGO" in upper_tokens:
                series = "Hilux Vigo"
            else:
                series = "Hilux"
        elif model_clean:
            series = model_clean

    elif brand == "MAZDA":
        # Example: "CX-5", "BT-50" style
        m = re.search(r"(CX[- ]?\d|BT[- ]?\d+)", " ".join(upper_tokens))
        if m:
            series = m.group(1).replace(" ", "-").upper().title()
        else:
            series = model_clean

    else:
        # default: just reuse cleaned model as 'series'
        series = model_clean

    result["series"] = series

    # ---------- trim (whatever is left over) ----------
    IGNORE = {
        "SEDAN", "HATCHBACK", "PICKUP", "DOUBLE", "CAB", "AT", "MT",
        "SUV", "4DR", "5DR", "2WD", "4WD", "DIESEL", "HYBRID",
        "SPORT", "COUPE",
    }

    remaining = []
    for tok in upper_tokens[1:]:
        if tok in IGNORE:
            continue
        if tok.isdigit():
            continue
        remaining.append(tok)

    if remaining:
        # keep first few tokens as trim
        result["trim"] = " ".join(remaining[:3]).title()

    return pd.Series(result)


## apply the parser to df_master

In [7]:
parsed = df_master.apply(parse_model_fields, axis=1)

df_master["model_clean"] = parsed["model_clean"]
df_master["series"]      = parsed["series"]
df_master["trim"]        = parsed["trim"]

# quick check
df_master[["brand", "model_raw", "model", "model_clean", "series", "trim"]].sample(20, random_state=1)


,brand,model_raw,model,model_clean,series,trim
6202,PORSCHE,NaN,911 GT3,911 GT3,None,None
2402,HONDA,NaN,CR-V,CR-V,None,None
3875,FORD,NaN,MUSTANG,MUSTANG,None,None
1483,TOYOTA,NaN,COROLLA ALTIS,COROLLA ALTIS,None,None
1586,TOYOTA,NaN,HILUX REVO,HILUX REVO,None,None
724,BMW,NaN,SERIES 4,SERIES 4,None,None
2075,TOYOTA,NaN,YARIS,YARIS,None,None
5476,HYUNDAI,NaN,H-100,H-100,None,None
6465,SUZUKI,NaN,APV,APV,None,None
2255,HONDA,NaN,CITY,CITY,None,None


## brand guesser (English + Thai, incl.

In [8]:
def guess_brand_from_text(text):
    if pd.isna(text):
        return pd.NA

    txt = str(text).upper()

    BRAND_PATTERNS = {
        "TOYOTA":   ["TOYOTA", "โตโยต้า", "โตโยตา"],
        "HONDA":    ["HONDA", "ฮอนด้า", "ฮอนดา"],
        "MAZDA":    ["MAZDA", "มาสด้า", "มาสดา"],
        "NISSAN":   ["NISSAN", "นิสสัน"],
        "ISUZU":    ["ISUZU", "อีซูซุ"],
        "SUZUKI":   ["SUZUKI", "ซูซูกิ"],
        "FORD":     ["FORD", "ฟอร์ด"],
        "MITSUBISHI": ["MITSUBISHI", "มิตซูบิชิ", "มิตซู"],
        "MERCEDES": ["MERCEDES", "BENZ", "เบนซ์"],
        "BMW":      ["BMW", "บีเอ็ม", "บีเอ็มดับเบิลยู"],
        "MG":       ["MG", "เอ็มจี"],
        "VOLVO":    ["VOLVO", "วอลโว่", "โวลโว่"],
        "HYUNDAI":  ["HYUNDAI", "ฮุนได"],
        "KIA":      ["KIA", "เกีย"],
        "CHEVROLET": ["CHEVROLET", "เชฟโรเลต", "เชฟ"],
        "BYD":      ["BYD", "บีวายดี"],
        "NETA":     ["NETA", "เนต้า"],
        "SUBARU":   ["SUBARU", "ซูบารุ"],
        "UD":       ["UD"],
    }

    for brand, tokens in BRAND_PATTERNS.items():
        for tok in tokens:
            if tok.upper() in txt:
                return brand

    return pd.NA


## fill missing brands using the guesser

In [9]:
# 1) try model_raw
mask = df_master["brand"].isna()
df_master.loc[mask, "brand"] = (
    df_master.loc[mask, "model_raw"].apply(guess_brand_from_text)
)

# 2) still missing? try model
mask = df_master["brand"].isna()
df_master.loc[mask, "brand"] = (
    df_master.loc[mask, "model"].apply(guess_brand_from_text)
)

# see remaining rows with no brand
df_master[df_master["brand"].isna()].head()


,brand,model,model_raw,year,price_thb,mileage_km,color,gear_type,location,lot_num,...,num_bids,num_hall_bids,num_internet_bids,min_bid,max_bid,source,channel,model_clean,series,trim
6693,<NA>,SEAL DYNAMIC,SEAL DYNAMIC,2023.0,540000.0,28095.0,Black,A/T,กรุงเทพมหานคร,1015,...,2.0,2.0,0.0,520000.0,540000.0,motto,auction,SEAL DYNAMIC,SEAL DYNAMIC,Dynamic


## finalise the model column

In [10]:
df_master["model"] = df_master["model"].fillna(df_master["model_clean"])

df_master[["brand", "model_raw", "model", "model_clean"]].sample(20, random_state=2)


,brand,model_raw,model,model_clean
3914,FORD,NaN,RANGER,RANGER
721,HONDA,NaN,ACCORD,ACCORD
4466,MERCEDES-BENZ,NaN,CLA45,CLA45
5001,VOLVO,NaN,C30,C30
4616,BMW,NaN,320 d,320 d
3054,NISSAN,NaN,NOTE,NOTE
472,BENZ,NaN,C-CLASS,C-CLASS
829,PEUGEOT,NaN,408,408
4810,BMW,NaN,520 i,520 i
1345,TOYOTA,NaN,AVANZA,AVANZA


## add flags for data quality + build df_pricing

In [11]:
# treat 0 or NaN price as "no price"
df_master["has_price"] = (
    df_master["price_thb"].notna() & (df_master["price_thb"] > 0)
)

# treat 0 or NaN mileage as "no mileage"
df_master["has_mileage"] = (
    df_master["mileage_km"].notna() & (df_master["mileage_km"] > 0)
)

# basic required fields
df_master["has_brand"] = df_master["brand"].notna()
df_master["has_model"] = df_master["model"].notna()
df_master["has_year"]  = df_master["year"].notna()

# this is the key flag for anything pricing/deal-checker related
df_master["ok_for_pricing"] = (
    df_master["has_price"]
    & df_master["has_mileage"]
    & df_master["has_brand"]
    & df_master["has_model"]
    & df_master["has_year"]
)

# sanity check
df_master["ok_for_pricing"].value_counts()


ok_for_pricing
False    5530
True     1275
Name: count, dtype: int64

In [13]:
# keep only rows that are good enough for pricing models / deal checker
df_pricing = df_master[df_master["ok_for_pricing"]].copy()

df_pricing.shape


(1275, 31)

## save both master + pricing-only CSVs

In [14]:
MASTER_OUTPUT_PATH  = DATA_DIR / "master_cars_all_sources.csv"
PRICING_OUTPUT_PATH = DATA_DIR / "cars_for_pricing_only.csv"

# Save with UTF-8-SIG so Excel reads Thai correctly
df_master.to_csv(MASTER_OUTPUT_PATH, index=False, encoding="utf-8-sig")
df_pricing.to_csv(PRICING_OUTPUT_PATH, index=False, encoding="utf-8-sig")

MASTER_OUTPUT_PATH, PRICING_OUTPUT_PATH


(WindowsPath('data/master_cars_all_sources.csv'),
 WindowsPath('data/cars_for_pricing_only.csv'))